## 3. Data Type Optimization & Column Type Casting
Converting raw generic types (`object`, `float64`) into precise, memory-efficient types:
* **`'Int64'` (Nullable Integer)**: Applied to count columns containing missing values (`Foreign_Visitors`, `Registered_Hotels_Count`, `Reported_Tourist_Complaints`, `International_Passengers_Arrival`, ratings) to maintain integer counts rather than converting to floats like `48112.0`.
* **`'category'`**: Categorical variables (`State_UT`, `Tourism_Category`, `Star_Classification`, `Port_of_Exit`) to reduce RAM and speed up aggregations. Months are typed as **ordered categories**.
* **`'string'`**: Explicit string dtype for names, IDs, and geographic descriptions.

*Each dataset is processed in its own dedicated cell with before/after `.head()` verification.*


In [ ]:
import pandas as pd
import numpy as np

# Load raw datasets for optimization
df_district_footfall = pd.read_csv('../data/raw/01_district_wise_tourist_footfall_2022_2024.csv')
df_nidhi_hospitality = pd.read_csv('../data/raw/02_nidhi_hospitality_establishments_registry.csv')
df_asi_monuments = pd.read_csv('../data/raw/03_asi_protected_monuments_inventory.csv')
df_inbound_survey = pd.read_csv('../data/raw/04_inbound_tourist_survey_microdata_2023_2024.csv')
df_airport_traffic = pd.read_csv('../data/raw/05_airport_passenger_traffic_monthly_2022_2024.csv')
df_macro_overview = pd.read_csv('../data/raw/06_annual_macro_overview_2022_2024.csv')
df_fta_summary = pd.read_csv('../data/raw/07_country_wise_fta_summary_2022_2024.csv')
df_state_ut_summary = pd.read_csv('../data/raw/08_state_ut_consolidated_summary_2022_2023.csv')
print('Raw datasets loaded for Type Optimization.')


### 3.1 Optimize Data Types: `df_district_footfall`


In [110]:
print('Types BEFORE optimization:')
print(df_district_footfall[['District', 'State_UT', 'Year', 'Foreign_Visitors', 'Registered_Hotels_Count']].dtypes)

df_district_footfall = df_district_footfall.astype({
    'District': 'string',
    'State_UT': 'category',
    'Year': 'int16',
    'Domestic_Visitors': 'int64',
    'Foreign_Visitors': 'Int64',           # Nullable integer
    'Total_Visitors': 'int64',
    'Tourism_Category': 'category',
    'Registered_Hotels_Count': 'Int64',    # Nullable integer
    'Avg_Stay_Duration_Days': 'float64',
    'Tourist_Police_Station_Available': 'category',
    'Reported_Tourist_Complaints': 'Int64' # Nullable integer
})

print('\nTypes AFTER optimization:')
print(df_district_footfall[['District', 'State_UT', 'Year', 'Foreign_Visitors', 'Registered_Hotels_Count']].dtypes)

# Verify with .head()
print('\n--- df_district_footfall.head(3) ---')
display(df_district_footfall.head(3))


Types BEFORE optimization:
District                       str
State_UT                       str
Year                         int64
Foreign_Visitors           float64
Registered_Hotels_Count    float64
dtype: object

Types AFTER optimization:
District                     string
State_UT                   category
Year                          int16
Foreign_Visitors              Int64
Registered_Hotels_Count       Int64
dtype: object

--- df_district_footfall.head(3) ---


,District,State_UT,Year,Domestic_Visitors,Foreign_Visitors,Total_Visitors,Tourism_Category,Registered_Hotels_Count,Avg_Stay_Duration_Days,Tourist_Police_Station_Available,Reported_Tourist_Complaints
0,Agra,Uttar Pradesh,2022,2133912,48112,2182024,Business_Urban,401,3.6,Yes,2
1,Agra,Uttar Pradesh,2023,2439018,54991,2494009,Business_Urban,401,1.3,Yes,34
2,Agra,Uttar Pradesh,2024,2308790,52054,2360844,Business_Urban,401,4.1,Yes,27


### 3.2 Optimize Data Types: `df_nidhi_hospitality`


In [111]:
# Convert Pincode: float -> Int64 -> string (avoids '497236.0')
df_nidhi_hospitality['Pincode'] = df_nidhi_hospitality['Pincode'].astype('Int64').astype('string')

df_nidhi_hospitality = df_nidhi_hospitality.astype({
    'Establishment_ID': 'string',
    'Establishment_Name': 'string',
    'Accommodation_Type': 'category',
    'Star_Classification': 'category',
    'State_UT': 'category',
    'District_City': 'string',
    'Total_Rooms': 'int64',
    'Room_Tariff_Min_INR': 'int64',
    'Room_Tariff_Max_INR': 'Int64',        # Nullable int for fixed-price units
    'Has_Conference_Facility': 'category',
    'Eco_Green_Certified': 'category',
    'MoT_Approval_Status': 'category',
    'Audit_Quality_Score': 'float64'
})

print('Data types cast for df_nidhi_hospitality.')
# Verify with .head()
display(df_nidhi_hospitality[['Establishment_ID', 'Establishment_Name', 'Accommodation_Type', 'Star_Classification', 'Pincode', 'Room_Tariff_Min_INR', 'Room_Tariff_Max_INR']].head(3))


Data types cast for df_nidhi_hospitality.


,Establishment_ID,Establishment_Name,Accommodation_Type,Star_Classification,Pincode,Room_Tariff_Min_INR,Room_Tariff_Max_INR
0,NIDHI-IND-00001,Heritage Chandigarh Suites,Guest House,Unclassified,497236,2500,<NA>
1,NIDHI-IND-00002,Mountain Khurda Boutique Stay,Homestay,NaN,543168,2500,6500
2,NIDHI-IND-00003,Mountain Bhagalpur Palace,Heritage Hotel,3 Star,244980,7500,11500


### 3.3 Optimize Data Types: `df_asi_monuments`


In [112]:
df_asi_monuments = df_asi_monuments.astype({
    'Monument_ID': 'string',
    'Monument_Name': 'string',
    'ASI_Circle': 'category',
    'State_UT': 'category',
    'District': 'string',
    'Location_Setting': 'category',
    'Era_Period': 'category',
    'Architectural_Type': 'category',
    'Is_Ticketed': 'category',
    'Entry_Fee_Domestic_INR': 'Int64',          # Nullable int (NaN for unticketed)
    'Entry_Fee_Foreign_INR': 'Int64',           # Nullable int (NaN for unticketed)
    'UNESCO_Heritage_Status': 'category',
    'Estimated_Annual_Footfall_2023': 'Int64',  # Nullable int
    'Estimated_Annual_Footfall_2025': 'Int64',  # Nullable int
    'Estimated_Annual_Footfall_2026': 'Int64',  # Nullable int
    'Conservation_Condition': 'category'
})

print('Data types cast for df_asi_monuments.')
# Verify with .head()
display(df_asi_monuments[['Monument_ID', 'Monument_Name', 'Architectural_Type', 'Is_Ticketed', 'Entry_Fee_Domestic_INR', 'Estimated_Annual_Footfall_2026']].head(3))


Data types cast for df_asi_monuments.


,Monument_ID,Monument_Name,Architectural_Type,Is_Ticketed,Entry_Fee_Domestic_INR,Estimated_Annual_Footfall_2026
0,ASI-IND-0001,Taj Mahal,Stupa/Monastery,Yes,40,7738564
1,ASI-IND-0002,Qutub Minar,Temple/Shrine,Yes,25,7621253
2,ASI-IND-0003,Red Fort,Fort/Bastion,Yes,50,3592065


### 3.4 Optimize Data Types: `df_inbound_survey`


In [113]:
month_order = [
    'January', 'February', 'March', 'April', 'May', 'June',
    'July', 'August', 'September', 'October', 'November', 'December'
]

# Ordered month category for seasonal comparisons
df_inbound_survey['Survey_Month'] = pd.Categorical(
    df_inbound_survey['Survey_Month'], categories=month_order, ordered=True
)

df_inbound_survey = df_inbound_survey.astype({
    'Response_ID': 'string',
    'Survey_Year': 'int16',
    'Port_of_Exit': 'category',
    'Tourist_Nationality': 'category',
    'Region_of_Origin': 'category',
    'Age_Group': 'category',
    'Gender': 'category',
    'Travel_Companion_Type': 'category',
    'Primary_Trip_Purpose': 'category',
    'Total_Nights_Stayed': 'int64',
    'Primary_State_Visited': 'category',
    'Estimated_Total_Spend_USD': 'float64',
    'Daily_Spend_INR': 'float64',
    'Booking_Channel': 'category',
    'Safety_Perception_Rating_1to5': 'Int64',
    'Overall_Experience_Score_1to10': 'Int64',
    'Would_Revisit_India': 'category'
})

print('Data types cast for df_inbound_survey.')
# Verify with .head()
display(df_inbound_survey[['Response_ID', 'Survey_Year', 'Survey_Month', 'Tourist_Nationality', 'Total_Nights_Stayed', 'Estimated_Total_Spend_USD', 'Safety_Perception_Rating_1to5']].head(3))


Data types cast for df_inbound_survey.


,Response_ID,Survey_Year,Survey_Month,Tourist_Nationality,Total_Nights_Stayed,Estimated_Total_Spend_USD,Safety_Perception_Rating_1to5
0,SURV-2024-00001,2023,September,USA,34,7628.0,2
1,SURV-2024-00002,2023,December,Japan,14,3987.0,<NA>
2,SURV-2024-00003,2023,March,UK,37,9184.0,2


### 3.5 Optimize Data Types: `df_airport_traffic`


In [114]:
df_airport_traffic['Month'] = pd.Categorical(
    df_airport_traffic['Month'], categories=month_order, ordered=True
)
# Add chronological monthly datetime timestamp
df_airport_traffic['Date'] = pd.to_datetime(
    df_airport_traffic['Year'].astype(str) + '-' + df_airport_traffic['Month'].astype(str) + '-01'
)

df_airport_traffic = df_airport_traffic.astype({
    'Airport_IATA': 'category',
    'Airport_Name': 'string',
    'City': 'string',
    'State_UT': 'category',
    'Year': 'int16',
    'Airport_Category': 'category',
    'Domestic_Passengers_Arrival': 'int64',
    'Domestic_Passengers_Departure': 'int64',
    'International_Passengers_Arrival': 'Int64',   # Nullable int for domestic airfields
    'International_Passengers_Departure': 'Int64', # Nullable int
    'Total_Aircraft_Movements': 'int64',
    'Cargo_Metric_Tonnes': 'float64',
    'On_Time_Departure_Pct': 'float64'
})

print('Data types cast for df_airport_traffic.')
# Verify with .head()
display(df_airport_traffic[['Airport_IATA', 'City', 'Date', 'Domestic_Passengers_Arrival', 'International_Passengers_Arrival', 'Total_Aircraft_Movements']].head(3))


Data types cast for df_airport_traffic.


,Airport_IATA,City,Date,Domestic_Passengers_Arrival,International_Passengers_Arrival,Total_Aircraft_Movements
0,DEL,Delhi,2022-01-01,2861195,1196281,55184
1,DEL,Delhi,2022-02-01,2926877,1119013,61310
2,DEL,Delhi,2022-03-01,2736173,950144,54210


### 3.6 Optimize Data Types: `df_macro_overview`, `df_fta_summary`, `df_state_ut_summary`


In [115]:
# Macro Overview
df_macro_overview = df_macro_overview.astype({
    'Year': 'int16',
    'Foreign_Tourist_Arrivals_FTA': 'int64',
    'International_Tourist_Arrivals_ITA': 'int64',
    'FTA_YoY_Growth_Pct': 'float64',
    'Foreign_Exchange_Earnings_INR_Crore': 'int64',
    'FEE_USD_Billion': 'float64',
    'Outbound_Indian_Departures': 'int64',
    'Outbound_Expenditure_USD_Billion': 'float64',
    'Tourism_Share_in_National_GDP_Pct': 'float64'
})

# Country FTA Summary
df_fta_summary = df_fta_summary.astype({
    'Country_of_Nationality': 'string',
    'Region': 'category',
    'FTA_2022': 'Int64', 'FTA_2023': 'Int64', 'FTA_2024': 'Int64', 'FTA_2025': 'Int64', 'FTA_2026': 'Int64',
    'Growth_2023_over_2022_Pct': 'float64', 'Growth_2024_over_2023_Pct': 'float64',
    'Growth_2025_over_2024_Pct': 'float64', 'Growth_2026_over_2025_Pct': 'float64',
    'Market_Share_2024_Pct': 'float64', 'Market_Share_2026_Pct': 'float64'
})

# State/UT Summary
df_state_ut_summary = df_state_ut_summary.astype({
    'State_UT': 'string',
    'Region': 'category',
    'Domestic_Visitors_2022': 'int64', 'Domestic_Visitors_2023': 'int64', 'Domestic_Visitors_2024': 'int64',
    'Domestic_Visitors_2025': 'int64', 'Domestic_Visitors_2026': 'int64',
    'Domestic_YoY_Growth_Pct': 'float64',
    'Foreign_Visitors_2022': 'int64', 'Foreign_Visitors_2023': 'int64', 'Foreign_Visitors_2024': 'int64',
    'Foreign_Visitors_2025': 'int64', 'Foreign_Visitors_2026': 'int64',
    'Foreign_YoY_Growth_Pct': 'float64',
    'Total_Visitors_2023': 'int64', 'Total_Visitors_2026': 'int64',
    'Foreign_Visitor_Share_2023_Pct': 'float64', 'Foreign_Visitor_Share_2026_Pct': 'float64'
})

print('Data types cast for macro, country, and state summaries.')
# Verify with .head()
display(df_macro_overview.head(3))
display(df_fta_summary[['Country_of_Nationality', 'Region', 'FTA_2024', 'FTA_2026', 'Market_Share_2026_Pct']].head(3))


Data types cast for macro, country, and state summaries.


,Year,Foreign_Tourist_Arrivals_FTA,International_Tourist_Arrivals_ITA,FTA_YoY_Growth_Pct,Foreign_Exchange_Earnings_INR_Crore,FEE_USD_Billion,Outbound_Indian_Departures,Outbound_Expenditure_USD_Billion,Tourism_Share_in_National_GDP_Pct
0,2022,6440000,13921000,NaN,177703,21.4,20900000,14.2,4.62
1,2023,9520000,18891000,47.83,230000,27.7,25300000,17.8,5.00
2,2024,9950000,20570000,4.52,293033,34.3,27100000,19.5,5.22


,Country_of_Nationality,Region,FTA_2024,FTA_2026,Market_Share_2026_Pct
0,USA,Americas,1804586,2113471,16.94
1,Bangladesh,South Asia,1750165,2100762,16.83
2,UK,Europe,1022587,1282855,10.28
